This notebook is responsible for generating COCO RLE ANNOTATIONS from spheric, color-masks, it is run SEPARATELY to creating images for dataset, and lasts veeery long.
There are many assumptions about processing masks, in respect to min_area_size, smoothing mask, retrieving pixel value.

In [1]:
import os

os.getcwd()

'c:\\Users\\WA\\Desktop\\badanie\\syncity3D\\jupyter'

In [1]:
import os
import re
import json
import shutil
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
from PIL import Image
from skimage.measure import label, regionprops
from pycocotools import mask as mask_utils
from tqdm import tqdm
from skimage.segmentation import expand_labels


In [ ]:
import cv2

@dataclass(frozen=True)
class ClassDef:
    name: str
    category_id: int


CITY_PARAMETERS_DIR = "C:\\Users\\WA\\Desktop\\badanie\\syncity3D\\COCO_DATASETS\\D3\\city_parameters.json"
with open (CITY_PARAMETERS_DIR, mode="r") as f:
    CITY_PARAMETERS = json.load(f)
#print(CITY_PARAMETERS)
PARAMETERS = CITY_PARAMETERS["parameters"]

# Ile klas?? tyle ile jest parametrów w mieście czy wszystkie możliwości?
#CLASS_DICT = {f"{param_value}": id for id, param_value in enumerate(np.arange(start=0, stop=255, step=1), start=1)}
CLASS_DICT = {f"{param_value}": id for id, param_value in enumerate([int(param*100) for param in PARAMETERS], start=1)}

CLASSES = [ ClassDef("Facade", 1000), ClassDef("Window", 1001) ] + [ClassDef(key, CLASS_DICT[key]) for key in CLASS_DICT.keys()]


def get_class_id_per_value(parameter_value: float) -> int:
    return CLASS_DICT.get(f"{parameter_value}", int(parameter_value))


def key_from_filename(fname: str, prefix: str) -> str:
    """
    Extract key from 'x_<id>_<angle>.<ext>' or 'y_<id>_<angle>.<ext>' -> '<id>_<angle>'

    Extract id from {prefix}_SynCity3D_<id>.<ext> -> <id> ??
    """
    base = os.path.basename(fname)
    if not base.startswith(prefix):
        raise ValueError(f"File {base} does not start with {prefix}")
    stem = os.path.splitext(base)[0]
    return stem.split("_")[-1].split("\.")[0]


def find_pairs(raw_dir: str) -> List[Tuple[str, str, str, str]]:
    """
    Returns list of (key, img_path, mask_path_m1, mask_path_m2)
    """

    imgs = []
    masks_m1 = []
    masks_m2 = []

    for f in os.listdir(raw_dir):
        if f.startswith("m-Normal"):
            imgs.append(f)
        elif f.startswith("m-1"):
            masks_m1.append(f)
        elif f.startswith("m-2"):
            masks_m2.append(f)

    img_map = {key_from_filename(f, "m-Normal"): os.path.join(raw_dir, f) for f in imgs}
    mask_map_m1 = {key_from_filename(f, "m-1"): os.path.join(raw_dir, f) for f in masks_m1}
    mask_map_m2 = {key_from_filename(f, "m-2"): os.path.join(raw_dir, f) for f in masks_m2}

    keys = sorted(set(img_map.keys()) & set(mask_map_m1.keys()) & set(mask_map_m2.keys()))
    
    if not keys:
        raise RuntimeError("No matching x_<id>_<angle> and y_<id>_<angle> pairs found.")

    triad = [(k, img_map[k], mask_map_m1[k], mask_map_m2[k]) for k in keys]
    return triad


def rgb_masks_from_mask_image(mask_rgb: np.ndarray, thr: int = 200) -> Dict[str, np.ndarray]:
    """
    Class separation from RGB mask using dominant channel rule.
    - poprawny: green-dominant pixels
    - niepoprawny: blue-dominant pixels
    thr limits near-white/gray pixels that otherwise contaminate masks.
    """
    if mask_rgb.ndim == 2:
        mask_rgb = np.repeat(mask_rgb[:, :, None], 3, axis=2)

    print("MASK_RGB")
    img = Image.fromarray(mask_rgb)
    img.show()


    R = mask_rgb[:, :, 0].astype(np.int16)
    G = mask_rgb[:, :, 1].astype(np.int16)
    B = mask_rgb[:, :, 2].astype(np.int16)

    poprawny = (R > 0)
    niepoprawny = (R == 0)

    return {"poprawny": poprawny, "niepoprawny": niepoprawny}


def instances_from_binary_mask(bin_mask: np.ndarray, min_area: int = 1) -> List[np.ndarray]:
    """
    Split a binary mask into instance masks using connected components.
    Returns list of HxW boolean masks.
    """
    lab = label(bin_mask.astype(np.uint8), connectivity=2)
    inst = []
    for r in regionprops(lab):
        if r.area < min_area:
            continue
        m = (lab == r.label)
        inst.append(m)
    return inst


def instances_from_channel_values(mask_rgb: np.ndarray, channel: int, thr: int = 1, min_area: int = 1):
    """
    Extract instance masks using unique channel values.
    Assumption: each instance is encoded with a unique intensity in a single channel.
    channel: 1 for green, 2 for blue
    """
    if mask_rgb.ndim == 2:
        mask_rgb = np.repeat(mask_rgb[:, :, None], 3, axis=2)

    ch = mask_rgb[:, :, channel].astype(np.uint8)

    vals = np.unique(ch)
    vals = vals[vals > thr]  # ignore background (0) and tiny noise

    inst = []
    for v in vals:
        m = (ch == v)
        if m.sum() < min_area:
            continue
        inst.append(m)
    return inst


def keep_only_pure_red(mask_rgb: np.ndarray) -> np.ndarray:
    """
    Keep only pixels that are exactly pure green of the form [0, G, 0] with G>0.
    Everything else is set to background [0,0,0].

    This removes obstacles [0,0,128], any anti-aliased edges, and any non-canonical colors.
    """
    if mask_rgb.ndim == 2:
        mask_rgb = np.repeat(mask_rgb[:, :, None], 3, axis=2)

    M = mask_rgb.astype(np.uint8)
    R = M[:, :, 0]
    G = M[:, :, 1]
    B = M[:, :, 2]

    pure_red = (R > 0) & (G == 0) & (B == 0)
    pure_green = (R == 0) & (G > 0) & (B == 0)


    out = np.zeros_like(M)
    out[pure_red, 0] = R[pure_red]  # keep original R id
    # we keep green channel too, as some facades can be extremely wiiide, so it w/h ratio exceeds 2.55
    out[pure_green, 1] = G[pure_green]

    return out


def clean_rgb_mask(mask_rgb: np.ndarray) -> np.ndarray:
    # in no case we mix 3 channels at once, so we can cut pixels which match this condition (these are potential artefacts)
    M = mask_rgb.astype(np.uint8)
    R = M[:, :, 0]
    G = M[:, :, 1]
    B = M[:, :, 2]

    artefacts = (R > 0) & (G > 0) & (B > 0)
    binary_mask = np.stack([artefacts, artefacts, artefacts], axis=2)
    mask_rgb[binary_mask] = 0   

    return mask_rgb


def filter_channels(mask_rgb: np.ndarray, channels = "rgb") -> np.ndarray:
    """
    Keep only pixels that are exactly pure channel
    Everything else is set to background [0,0,0].

    """

    if mask_rgb.ndim == 2:
        mask_rgb = np.repeat(mask_rgb[:, :, None], 3, axis=2)


    M = mask_rgb.astype(np.uint8)
    R = M[:, :, 0]
    G = M[:, :, 1]
    B = M[:, :, 2]

    # yeah, mixing channels was a bad idea, 'pure'
    pure_red = (R > 0) & (G == 0) & (B == 0)
    pure_green = (G > 0) & (B == 0) & (R == 0)
    pure_blue = (B > 0) & (R == 0) & (G == 0)

    out = np.zeros_like(M)

    if "r" in channels:
        out[pure_red, 0] = R[pure_red]  # keep original R id
    # we keep green channel too, as some facades can be extremely wiiide, so it w/h ratio exceeds 2.55
    if "g" in channels:
        out[pure_green, 1] = G[pure_green]

    if "b" in channels:
        out[pure_blue, 2] = B[pure_blue]

    #Image.fromarray(out).show()

    return out


def facade_instances_from_green_ids(mask_rgb_clean: np.ndarray, min_area: int = 1):
    """
    Extract facade instances assuming:
      - facades are encoded as [0, G, 0] (pure green)
      - instance id = unique G value
    """
    M = mask_rgb_clean.astype(np.uint8)
    G = M[:, :, 1]
    # only values that actually appear (excluding 0 background)
    ids = np.unique(G)
    ids = ids[ids > 0]

    inst = []
    for v in ids:
        m = (G == v)
        if int(m.sum()) >= min_area:
            inst.append(m)
    return inst


def coco_rle_from_mask(mask: np.ndarray) -> Dict:
    """
    Use pycocotools to produce COCO RLE.
    mask must be Fortran-contiguous (column-major) for correct encoding.
    """
    m = np.asfortranarray(mask.astype(np.uint8))
    rle = mask_utils.encode(m)
    # pycocotools returns counts as bytes; convert to utf-8 string for JSON
    rle["counts"] = rle["counts"].decode("utf-8")
    
    return rle


def canonicalize_facades(mask_rgb: np.ndarray, tol: int = 20, thr_soft: int = 30, max_expand: int = 5) -> np.ndarray:
    """
    Produces a clean mask where facades are encoded as [0,G,0] with stable instance ids.
    - Seeds: exact pixels [0,G,0] (R==0,B==0,G>0)
    - Soft facade pixels: green-dominant near-pure pixels (handles anti-aliasing)
    - Expansion assigns soft pixels to nearest seed (instance), up to max_expand pixels.
    """
    M = mask_rgb.astype(np.uint8)
    R, G, B = M[:, :, 0], M[:, :, 1], M[:, :, 2]

    # 1) exact seeds for each instance id
    seed = (R == 0) & (B == 0) & (G > 0)
    label_img = np.zeros(G.shape, dtype=np.int32)

    # label by G value (instance id)
    # Note: assumes G values are manageable; if many, this is still OK for your sizes
    ids = np.unique(G[seed])
    for idx, v in enumerate(ids, start=1):
        label_img[seed & (G == v)] = idx

    # 2) soft facade pixels around edges (tolerant rule)
    soft = (G >= thr_soft) & (G > R) & (G > B) & (R <= tol) & (B <= tol)

    # 3) expand labels to cover soft pixels
    # expand_labels grows labels outward; we then keep labels only where soft is True
    expanded = expand_labels(label_img, distance=max_expand)
    expanded[~soft] = 0

    # 4) rebuild canonical RGB mask: [0, G_id, 0]
    out = np.zeros_like(M)
    # map back: label index -> original G value
    # build lookup array
    id_lookup = np.zeros((len(ids) + 1,), dtype=np.uint8)
    for idx, v in enumerate(ids, start=1):
        id_lookup[idx] = v

    out[:, :, 1] = id_lookup[expanded]
    return out


def bbox_from_mask(mask: np.ndarray) -> List[float]:
    ys, xs = np.where(mask)
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    return [float(x0), float(y0), float(x1 - x0 + 1), float(y1 - y0 + 1)]


def get_parameter_value_from_instance_mask(instance_binary_mask: np.ndarray, rgb_mask: np.ndarray) -> float:
    

    #instance_mask = np.repeat(instance_binary_mask[:, :, None], 3, axis=2)
    #instance_mask = np.stack([instance_binary_mask, instance_binary_mask, instance_binary_mask], axis=2)

    # mask with dark pixels
    #instance_idx = (instance_binary_mask == 0)
    rgb_mask_filtered = rgb_mask.copy()
    rgb_mask_filtered[~instance_binary_mask] = 0

    #Image.fromarray(instance_binary_mask).show()
    #Image.fromarray(rgb_mask_filtered).show()

    # sum because proportion parameter can comprise R and G channel in M1, in M2 channels are separated, so it doesn't affect anything
    #accumulated_mask = np.sum(rgb_mask_filtered, axis=2)
    M = rgb_mask_filtered.astype(np.uint8)
    R = M[:, :, 0]

    values, counts = np.unique(R, return_counts=True)

    #Image.fromarray(instance_binary_mask).show()
    #Image.fromarray(rgb_mask_filtered).show()

    print()
    print("VALUES:", values)
    print("COUNTS:", counts)

    #skip instances
    if len(values) == 1:
        return -1 

    most_common_value = np.argmax(counts[1:]) # we exclude dark/mask (0) at the start
    true_value = values[most_common_value + 1]

    print("MOST COMMON VALUES:", most_common_value)
    print("TRUE PARAM VALUE:", true_value)

    match = find_closest_match(true_value)
    print(f"CLOSEST MATCH TO {true_value} is {match}")

    return int(match)


def find_closest_match(true_value):
    #print(PARAMETERS)
    distances = [abs(true_value-param * 100) for param in PARAMETERS]
    #print("Distances", distances)
    min_distance = np.argmin(np.array(distances))
    match = PARAMETERS[min_distance]
    return match * 100 # return pixel value


def smooth_binary_mask(instance_binary_mask: np.ndarray) -> np.ndarray:
    '''
    Smoothing BINARY masks, because initial binary mask suffer heavily from aliasing in previous step (any value in channel > 0),
    also rendering in CityEngine yields some artifacts.

    We basically apply basic smoothering ! and expand 1 pixel -- I am not sure why, i just copied it from canonicalize facades
    '''
    Image.fromarray(instance_binary_mask).show()

    kernel = np.ones((7,7), np.uint8)

    mask_cv2 = (instance_binary_mask * 255).astype(np.uint8)
    cleaned = cv2.morphologyEx(mask_cv2, cv2.MORPH_CLOSE, kernel)
    
    # NOTE: NOT QUITE SURE WHAT TO SOFTEN IN BINARY MASK
    # 2) soft facade pixels around edges (tolerant rule)
    #soft = (cleaned >= thr_soft) & (R <= tol) & (B <= tol)

    # 3) expand labels to cover soft pixels
    # expand_labels grows labels outward 1 pixel
    expanded = expand_labels(cleaned, distance=1)
    #expanded[~soft] = 0
    Image.fromarray(expanded).show()

    return expanded


def export_split_to_coco(
    pairs: List[Tuple[str, str, str, str]],
    out_images_dir: str,
    out_json_m1_path: str,
    out_json_m2_path: str,
    copy_images: bool = False,
    thr: int = 200,
    min_facade_area: int = 100,
    min_window_area: int = 100
):
    os.makedirs(out_images_dir, exist_ok=True)
    os.makedirs(os.path.dirname(out_json_m1_path), exist_ok=True)
    os.makedirs(os.path.dirname(out_json_m2_path), exist_ok=True)
    
    images = []
    annotations_m1 = []
    annotations_m2 = []

    categories = [{"id": c.category_id, "name": c.name, "supercategory": "object"} for c in CLASSES]

    ann_id = 1
    ann_m2_id = 1

    # visual test is flawed as it displays rigid mask (not smoothed) which are later registered in segmentation RLE
    visual_test = False

    for img_id, (key, img_path, mask_m1_path, mask_m2_path) in enumerate(tqdm(pairs, desc=f"Exporting annotations"), start=1):
        # Link/copy image
        img_name = os.path.basename(img_path)
        out_img_path = os.path.join(out_images_dir, img_name)

        if not os.path.exists(out_img_path):
            if copy_images:
                shutil.copy2(img_path, out_img_path)
            else:
                # symlink if supported; fallback to copy
                try:
                    os.symlink(os.path.abspath(img_path), out_img_path)
                except OSError:
                    shutil.copy2(img_path, out_img_path)

        with Image.open(img_path) as im:
            w, h = im.size

        images.append({"id": img_id, "file_name": img_name, "width": w, "height": h})

        if visual_test:
            Image.open(img_path).show()


        #Image.open(img_path).show()
        #
        # METHOD 1 ====================================================================================
        #
        #print("\nMETHOD 1")

        mask_m1_rgb = np.array(Image.open(mask_m1_path).convert("RGB"))

        # just in case
        mask_m1_rgb = clean_rgb_mask(mask_m1_rgb)

        # - VISUAL TEST MIN AREA M1
        if visual_test:
            Image.fromarray(mask_m1_rgb).show()
        # - VISUAL TEST MIN AREA M1
        
        
        # 1) keep RED and GREEN pixels [R,<G>,0]  - these pixels represent FACADES in M1
        facade_mask_m1 = filter_channels(mask_m1_rgb, channels="r")
        
        # 2) binary mask: any RED pixel = facade (but value of a parameter sometimes can be carried also in GREEN, but WITHIN the RED boundaries)
        facade_mask_m1_filter = facade_mask_m1.copy()
        #print(facade_mask_m1.shape)
        facade_binary_m1 = (facade_mask_m1_filter[:, :, 0] > 0)

        #Image.fromarray(facade_binary_m1).show()
        
        # - Found facades
        # 3) connected components → handles duplicate colors (wymaganie b)
        #    min_area filters small instances (wymaganie a)
        facade_instances_m1 = instances_from_binary_mask(facade_binary_m1, min_area=min_facade_area)
        #print("HOW MANY INSTANCES:", len(facade_instances_m1))

        # - VISUAL TEST MIN AREA M1
        if visual_test:
            visual_test_area_mask = np.zeros(shape=facade_binary_m1.shape)
        # - VISUAL TEST MIN AREA M1

        # 4) write annotations: only category_id=1 ("poprawny"/facade)
        for instance_mask in facade_instances_m1[:1]:

            #Image.fromarray(instance_mask).show()
            instance_mask_smooth = smooth_binary_mask(instance_mask)
            
            # - VISUAL TEST MIN AREA M1
            if visual_test:
                instance_mask_bool = (instance_mask_smooth > 0)
                visual_test_area_mask[instance_mask_bool] = 255
            # - VISUAL TEST MIN AREA M1

            #Image.fromarray(instance_mask_smooth).show()
            #Image.fromarray(instance_mask).show()

            rle = coco_rle_from_mask(instance_mask_smooth)
            area = float(mask_utils.area(rle))
            #print("AREA:", area)
            bbox = mask_utils.toBbox(rle).tolist()
            parameter_value = get_parameter_value_from_instance_mask(instance_mask, facade_mask_m1)
            
            #print(instance_mask.shape, instance_mask.dtype)
            #print(instance_mask_smooth.shape, instance_mask_smooth.dtype)

            annotations_m1.append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": get_class_id_per_value(parameter_value),     # parameter value
                "segmentation": rle,
                "area": area,
                "bbox": bbox,
                "iscrowd": 0,
                "parameter_value": parameter_value      #just leave it for now, 
            })

            ann_id += 1

        # - VISUAL TEST MIN AREA M1
        if visual_test:
            visual_test_area_mask_bool = (visual_test_area_mask > 0)
            visual_test_mask = np.stack([visual_test_area_mask_bool, visual_test_area_mask_bool, visual_test_area_mask_bool], axis=2)
            facade_mask_m1_copy = facade_mask_m1.copy()
            facade_mask_m1_copy[~visual_test_mask] = 0

            Image.fromarray(facade_mask_m1_copy).show()
        # - VISUAL TEST MIN AREA M1

        #        
        # METHOD 2 =====================================================================
        #

        #print("\nMETHOD 2")
        
        mask_m2_rgb = np.array(Image.open(mask_m2_path).convert("RGB"))
        # just in case
        mask_m2_rgb = clean_rgb_mask(mask_m2_rgb)
       
       # Find Facades - this time, they are represented in a fixed color value (~255) on a different channel - wheter G or B is arbirtrary, we chose: B
        mask_facades_m2 = filter_channels(mask_m2_rgb, "rb")
        mask_windows_m2 = filter_channels(mask_m2_rgb, "r")

        # - VISUAL TEST MIN AREA M2
        if visual_test:
            mask_image_rgb = filter_channels(mask_m2_rgb, "rb")
            Image.fromarray(mask_image_rgb).show()
        # - VISUAL TEST MIN AREA M2

        facade_binary_m2 = (mask_facades_m2[:,:, 2] > 0)
        windows_binary_m2 = (mask_windows_m2[:, :, 0] > 0)

        # FACADE ============

        facade_instances_m2 = instances_from_binary_mask(facade_binary_m2, min_area=min_facade_area)
        facade_instances_ids = []

        # - VISUAL TEST MIN AREA M2
        if visual_test:
            visual_test_area_mask = np.zeros(shape=facade_binary_m2.shape)
        # - VISUAL TEST MIN AREA M2

        for idx, facade_instance in enumerate(facade_instances_m2):

            #Image.fromarray(facade_instance).show()
            parameter_value = float(-1)
            facade_instance_smooth = smooth_binary_mask(facade_instance)
            rle = coco_rle_from_mask(facade_instance_smooth)
            area = float(mask_utils.area(rle))
            bbox = mask_utils.toBbox(rle).tolist()
        
            # - VISUAL TEST MIN AREA M2
            if visual_test:
                instance_mask_bool = (facade_instance_smooth > 0)
                visual_test_area_mask[instance_mask_bool] = 255
            # - VISUAL TEST MIN AREA M2

        
            #Image.fromarray(facade_instance_smooth).show()
            
            # WINDOWS ============

            # outer boundaries
            contour, hierarchy = cv2.findContours(facade_instance_smooth, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

            facade_filter_mask = np.zeros_like(facade_instance_smooth)
            # fill outer contour
            cv2.drawContours(facade_filter_mask, contour, -1, 255, thickness=-1)

            binary_facade_filter_mask = (facade_filter_mask > 0)

            windows_binary_mask_filtered = windows_binary_m2.copy()
            windows_binary_mask_filtered[~binary_facade_filter_mask] = 0

            # mini-optimization - parameter is the same for every window instance
            #Image.fromarray(windows_binary_mask_filtered).show(title="Windows Binary Mask Filtered")
            #Image.fromarray(mask_windows_m2).show(title="Mask Windows M2")


            parameter_value = get_parameter_value_from_instance_mask(windows_binary_mask_filtered, mask_windows_m2)
            if parameter_value == -1:
                continue
            
            # annotations representing a Facade
            annotations_m2.append({
                "id": ann_m2_id,
                "image_id": img_id,
                "category_1_id": 1000,     # semantics: Facade 
                "category_2_id": 1000,     # Dummy value
                "segmentation": rle,
                "area": area,
                "bbox": bbox,
                "iscrowd": 0,
                "parameter_value": parameter_value,
                "inside_area": -1
            })
            facade_instances_ids.append(ann_m2_id)
            ann_m2_id += 1


            windows_instances = instances_from_binary_mask(windows_binary_mask_filtered, min_area=min_window_area)

            for id_w, windows_instance in enumerate(windows_instances):
                windows_instance_smooth = smooth_binary_mask(windows_instance)
                rle = coco_rle_from_mask(windows_instance_smooth)
                area = float(mask_utils.area(rle))
                bbox = mask_utils.toBbox(rle).tolist()
                #parameter_value = get_parameter_value_from_instance_mask(windows_instance, mask_windows_m2)
                # - VISUAL TEST MIN AREA M2
                if visual_test:
                    instance_mask_bool = (windows_instance_smooth > 0)
                    visual_test_area_mask[instance_mask_bool] = 255
                # - VISUAL TEST MIN AREA M2

                #Image.fromarray(windows_instance).show()
                #Image.fromarray(windows_instance_smooth).show()
                
                # annotations representing a Window
                annotations_m2.append({
                    "id": ann_m2_id,
                    "image_id": img_id,
                    "category_1_id": 1001,     # semantics: Window 
                    "category_2_id": get_class_id_per_value(parameter_value),     # Window Parameter Value
                    "segmentation": rle,
                    "area": area,
                    "bbox": bbox,
                    "iscrowd": 0,
                    "parameter_value": parameter_value,
                    "inside_area": facade_instances_ids[-1]
                })

                ann_m2_id += 1
         
        # - VISUAL TEST MIN AREA M2
        if visual_test:
            visual_test_area_mask_bool = (visual_test_area_mask > 0)
            visual_test_mask = np.stack([visual_test_area_mask_bool, visual_test_area_mask_bool, visual_test_area_mask_bool], axis=2)
            mask_m2_copy = mask_image_rgb.copy()
            mask_m2_copy[~visual_test_mask] = 0

            Image.fromarray(mask_m2_copy).show()
        # - VISUAL TEST MIN AREA M2       
        
    #print("ANNOTATIONS M1:")
    #for ann in annotations_m1:
    #    print("A1:", ann)

    #print("ANNOTATIONS M2:")
    #for ann in annotations_m2:
    #    print("A2:", ann)

    
    coco_m1 = {"images": images, "annotations": annotations_m1, "categories": categories}
    coco_m2 = {"images": images, "annotations": annotations_m2, "categories": categories}

    with open(out_json_m1_path, "w", encoding="utf-8") as f:
        json.dump(coco_m1, f, ensure_ascii=False, indent=3)
    with open(out_json_m2_path, "w", encoding="utf-8") as f:
        json.dump(coco_m2, f, ensure_ascii=False, indent=3)

    return


def make_splits(pairs: List[Tuple[str, str, str]], seed: int = 0, train=0.8, val=0.1):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(pairs))
    rng.shuffle(idx)

    n = len(pairs)
    n_train = int(round(train * n))
    n_val = int(round(val * n))
    n_test = n - n_train - n_val

    train_pairs = [pairs[i] for i in idx[:n_train]]
    val_pairs = [pairs[i] for i in idx[n_train:n_train+n_val]]
    test_pairs = [pairs[i] for i in idx[n_train+n_val:]]

    return train_pairs, val_pairs, test_pairs


In [ ]:
print("START", "="*10)

RAW_DIR = "C:\\Users\\WA\\Desktop\\badanie\\syncity3D\\spheric_photos_and_masks"
OUT_DIR = "C:\\Users\\WA\\Desktop\\badanie\\syncity3D\\COCO_DATASETS\\D3" 

# test of 5 images x m-X
#RAW_DIR = "C:\\Users\WA\Desktop\\misc\\sample_dataset_new_dimmed"
#OUT_DIR = "C:\\Users\\WA\\Desktop\\misc\\sample_dataset_out_new_dimmed"

pairs = find_pairs(RAW_DIR)
train_pairs, val_pairs, test_pairs = make_splits(pairs, seed=0, train=0.8, val=0.1)

#print("test_pairs")
#print(test_pairs)

# filter small instances 
min_facade_area = 20000
min_windows_area = 100
copy_images=True


export_split_to_coco(
    train_pairs,
    out_images_dir=os.path.join(OUT_DIR, "images", "train"),
    out_json_m1_path=os.path.join(OUT_DIR, "annotations", "instances_train_m1.json"),
    out_json_m2_path=os.path.join(OUT_DIR, "annotations", "instances_train_m2.json"),
    copy_images=copy_images,
    thr=128,
    min_facade_area=min_facade_area,  
    min_window_area=min_windows_area
)

export_split_to_coco(
    val_pairs,
    out_images_dir=os.path.join(OUT_DIR, "images", "val"),
    out_json_m1_path=os.path.join(OUT_DIR, "annotations", "instances_val_m1.json"),
    out_json_m2_path=os.path.join(OUT_DIR, "annotations", "instances_val_m2.json"),
    copy_images=copy_images,
    thr=128,
    min_facade_area=min_facade_area, 
    min_window_area=min_windows_area
)

export_split_to_coco(
    test_pairs,
    out_images_dir=os.path.join(OUT_DIR, "images", "test"),
    out_json_m1_path=os.path.join(OUT_DIR, "annotations", "instances_test_m1.json"),
    out_json_m2_path=os.path.join(OUT_DIR, "annotations", "instances_test_m2.json"),
    copy_images=copy_images,
    thr=128,
    min_facade_area=min_facade_area,
    min_window_area=min_windows_area
)

# Zapis splitu dla reprodukowalności
split_path = os.path.join(OUT_DIR, "split_keys.json")
with open(split_path, "w", encoding="utf-8") as f:
    json.dump({
        "train": [k for (k, _, _, _) in train_pairs],
        "val": [k for (k, _, _, _) in val_pairs],
        "test": [k for (k, _, _, _) in test_pairs],
    }, f, ensure_ascii=False, indent=2)

print("DONE", "="*10)

START ==========


Exporting annotations: 100%|██████████| 1/1 [00:02<00:00,  2.06s/it]


VALUES: [  0 113 114 115 116 117 118 119 120 121 122 123]
COUNTS: [18081744        5       10       10       22       84       53       73
       74      109      365   400651]
MOST COMMON VALUES: 10
TRUE PARAM VALUE: 123
CLOSEST MATCH TO 123 is 124.0
DONE ==========
